# Classical Methods Comparison
## Tutorial 6: CBF vs Capon — Performance Analysis

A systematic performance comparison between Conventional Beamforming (CBF) and Capon (MVDR), evaluated across:

1. **Angular resolution** – how close can sources be?
2. **SNR sensitivity** – performance vs noise level
3. **Snapshot requirement** – performance vs $N$
4. **Bias and RMSE** – accuracy characterisation
5. **Monte Carlo evaluation**

---

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import sys, os

sys.path.insert(0, os.path.join(os.getcwd(), '..', 'src'))

from doa_methods.array_processing import UniformLinearArray, SignalModel
from doa_methods.classical import ConventionalBeamforming, CaponBeamforming

plt.style.use('default')
plt.rcParams['figure.figsize'] = (12, 7)
plt.rcParams['font.size'] = 12

M = 16
array = UniformLinearArray(M=M, d=0.5)
sm    = SignalModel(array)
cbf   = ConventionalBeamforming(array)
capon = CaponBeamforming(array, diagonal_loading=1e-3)

angle_grid = np.linspace(-np.pi/2, np.pi/2, 1801)

def rmse_trial(method, doas_true, N, snr_db, n_trials=200, seed_start=0):
    """Monte Carlo RMSE for a single (N, snr_db) pair."""
    K = len(doas_true)
    errors = []
    for trial in range(n_trials):
        X, _, _ = sm.generate_signals(doas_true, N, snr_db, seed=seed_start+trial)
        try:
            est = np.sort(method.estimate(X, K=K))
            true_s = np.sort(doas_true)
            errors.append(np.sqrt(np.mean((est - true_s)**2)))
        except Exception:
            errors.append(np.pi)   # failure
    return np.sqrt(np.mean(np.array(errors)**2))

print("Utility functions ready.")

## 1. Resolution vs Angular Separation

We define **resolution** as the minimum angular separation at which both sources produce clearly distinct peaks in the spectrum (both estimated DOAs within 1° of truth).

In [ ]:
doas_center = np.deg2rad(0)
separations = np.linspace(1, 20, 20)
snr_db      = 15
N           = 300
n_trials    = 100

rmse_cbf_sep   = []
rmse_capon_sep = []

for sep in separations:
    th1 = doas_center - np.deg2rad(sep/2)
    th2 = doas_center + np.deg2rad(sep/2)
    doas_test = np.array([th1, th2])
    rmse_cbf_sep.append(  np.rad2deg(rmse_trial(cbf,   doas_test, N, snr_db, n_trials)))
    rmse_capon_sep.append(np.rad2deg(rmse_trial(capon, doas_test, N, snr_db, n_trials)))

fig, ax = plt.subplots(figsize=(11, 6))
ax.semilogy(separations, rmse_cbf_sep,   'b-o', ms=6, lw=2, label='CBF')
ax.semilogy(separations, rmse_capon_sep, 'r-s', ms=6, lw=2, label='Capon')
ax.axhline(1.0, color='gray', ls=':', label='1° threshold')
rayleigh = np.rad2deg(0.886/((M-1)*0.5))
ax.axvline(rayleigh, color='b', ls='--', alpha=0.5,
           label=f'CBF Rayleigh {rayleigh:.1f}°')
ax.set_xlabel('Angular Separation (°)')
ax.set_ylabel('RMSE (°)')
ax.set_title(f'Resolution vs Separation  (M={M}, SNR={snr_db} dB, N={N})')
ax.legend(); ax.grid(True, alpha=0.3)
plt.tight_layout(); plt.show()

## 2. RMSE vs SNR

In [ ]:
doas_fixed = np.deg2rad([-20.0, 15.0])
snr_range  = np.arange(-5, 26, 3)
N_fixed    = 200
n_trials   = 150

rmse_cbf_snr   = [np.rad2deg(rmse_trial(cbf,   doas_fixed, N_fixed, s, n_trials)) for s in snr_range]
rmse_capon_snr = [np.rad2deg(rmse_trial(capon, doas_fixed, N_fixed, s, n_trials)) for s in snr_range]

fig, ax = plt.subplots(figsize=(11, 6))
ax.semilogy(snr_range, rmse_cbf_snr,   'b-o', ms=6, lw=2, label='CBF')
ax.semilogy(snr_range, rmse_capon_snr, 'r-s', ms=6, lw=2, label='Capon')
ax.set_xlabel('SNR (dB)')
ax.set_ylabel('RMSE (°)')
ax.set_title(f'RMSE vs SNR  (M={M}, N={N_fixed}, sources at –20° and 15°)')
ax.legend(); ax.grid(True, alpha=0.3)
plt.tight_layout(); plt.show()

## 3. RMSE vs Number of Snapshots

In [ ]:
snr_fixed   = 10
N_range     = [10, 20, 50, 100, 200, 500, 1000]
n_trials    = 150

rmse_cbf_N   = [np.rad2deg(rmse_trial(cbf,   doas_fixed, N, snr_fixed, n_trials)) for N in N_range]
rmse_capon_N = [np.rad2deg(rmse_trial(capon, doas_fixed, N, snr_fixed, n_trials)) for N in N_range]

fig, ax = plt.subplots(figsize=(11, 6))
ax.loglog(N_range, rmse_cbf_N,   'b-o', ms=6, lw=2, label='CBF')
ax.loglog(N_range, rmse_capon_N, 'r-s', ms=6, lw=2, label='Capon')
ax.axvline(M, color='gray', ls=':', label=f'N = M = {M}')
ax.set_xlabel('Number of Snapshots N')
ax.set_ylabel('RMSE (°)')
ax.set_title(f'RMSE vs Snapshots  (M={M}, SNR={snr_fixed} dB)')
ax.legend(); ax.grid(True, alpha=0.3)
plt.tight_layout(); plt.show()

## 4. Summary Table

In [ ]:
print("="*65)
print(f"{'Metric':<30} {'CBF':>15} {'Capon':>15}")
print("="*65)

# resolution threshold (first sep with RMSE < 1°)
def resolution_thresh(rmse_list, seps):
    for sep, r in zip(seps, rmse_list):
        if r < 1.0: return f'{sep:.1f}°'
    return '>20°'

print(f"{'Resolution threshold':<30} "
      f"{resolution_thresh(rmse_cbf_sep, separations):>15} "
      f"{resolution_thresh(rmse_capon_sep, separations):>15}")

print(f"{'RMSE @ SNR=10dB (°)':<30} "
      f"{rmse_cbf_snr[snr_range.tolist().index(10)]:>15.3f} "
      f"{rmse_capon_snr[snr_range.tolist().index(10)]:>15.3f}")

print(f"{'Min snapshots for RMSE<5° ':<30} ", end='')
for rmse_list in [rmse_cbf_N, rmse_capon_N]:
    for N_, r in zip(N_range, rmse_list):
        if r < 5.0:
            print(f"{N_:>15}", end=' ')
            break
    else:
        print(f"{'N/A':>15}", end=' ')
print()
print("="*65)

## Exercises
1. Repeat the RMSE vs SNR comparison for **three** sources and comment on whether the relative ranking changes.
2. Implement a simple **success rate** metric: fraction of trials where all estimated DOAs are within 2° of the true values.  Plot success rate vs SNR for both methods.
3. Add a third method — **windowed CBF** (Hann window) — to the comparison plots.  Where does it sit relative to uniform CBF and Capon?